# 25 — Milestone 3: the whole run (20 → 21 → 22 → 23 → 24 → 26)

Owner: Member 1. Runs every step of Milestone 3 in order, from the Milestone 1–2 data on Drive to the six
Hugging Face tables of each member, with the notebooks of the group. Their own tests are not run here: each
notebook runs them when it is opened on its own. **One run on Colab once the 20 manual sheets are in
`labels/manual/`.**

| Step | Notebook (owner) | Function | Reads | Writes |
|---|---|---|---|---|
| `data` | `20_m3_data` (TV1) | `build_m3_data()` | `data/parsed/`, `data/raw/*/references.json` | `data/m3/{entries,refs,pubs}.parquet`; nothing when they are up to date |
| `labels` | `21_labeling` (TV2) | `run_labeling()` | `labels/manual/*.csv`, `data/m3/` | `labels/auto/auto_labels.parquet`, `labels/split.json`, `labels/labels.parquet`, `logs/m3_labels.json` |
| `features` | `22_features` (TV3) | `run_features()` | `labels/labels.parquet`, `labels/split.json`, `data/m3/` | `data/m3/pairs.parquet`, `logs/m3_feature_report.csv`, figures |
| `model` | `23_model` (TV3) | `run_model()` | `data/m3/pairs.parquet` | `data/m3/predictions.parquet`, `logs/m3_metrics.json`, `logs/m3_model_selection.csv` |
| `analysis` | `24_analysis` (TV4) | `run_analysis()` | all of the above | `logs/m3_requirements.json`; skipped while 24 is not on this branch |
| `export` | `26_export_hf` (TV1) | `export_all()`, `check_export()` | `data/raw/`, `data/m2/`, `predictions` | `hf/<member>/` (six tables and the dataset card), `logs/hf_export.json` |

The upload is not a step: each member uploads their own `hf/<member>/` with their own token
(`26_export_hf`, `UPLOAD_MEMBER`), as the teacher asked.

**Record.** Every run appends to `logs/m3_run.json` the code commit, the parser, the signature of the `data/m3/`
build and, per step, its status, seconds and main numbers (labels, split, chosen model, MRR@5, exported rows).
The poster takes its numbers from the last complete run, the one that was exported.

**Order and freshness.** A step reads the files of the steps before it, so a run can start at any step
(`RUN_STEPS`). After the run, `stale_steps()` compares modification times and lists every step whose output
is older than one of its inputs (new manual sheets after `labels.parquet`, a new `predictions` after the
export): the numbers of the poster never mix two runs.

**While the labels are being made**, `RUN_LABEL_STATUS = True` only prints each member's progress (sheets in
`labels/manual/` against `labels/selection.csv`, then `validate_labels` without the final minimums) and writes
nothing.

**Namespaces.** `20`, `21`, `22`, `23` and `26` share one namespace (they were written to: `21` uses the paths of
`20`, `22` the helpers of `21`). `24` defines its own `member_of`, `MEMBER_RANGES`, `KEY`, `_FID_RE`, ... and is
meant to run alone, so `run_isolated` runs it in a namespace of its own, given `MEMBER_RANGES` of `20`.

## 0. Setup

**On Colab** (CPU runtime): open this notebook, keep the settings, *Run all*. The first cell clones the branch
and works from `src/`, like `15` and `20`. To run only some steps (after a disconnect, or after a member fixed a
sheet), list them in `RUN_STEPS`; the earlier steps' files on Drive are used. `RUN_AUTO_FRACTION` lowers the
share of automatically labelled publications when `22`/`23` take too long (`21` never goes below the 10% of
the statement). `M3_ROOT` points to another project folder (a local copy, a test folder). Another notebook
can set `RUN_OVERRIDES = {...}` before `%run -i` to change any value of the settings cell.

In [ ]:
# On Colab, when this notebook is opened on its own: clone or update the branch, install the packages and work
# from src/ (the notebooks use relative paths).
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB and not Path("./00_config.ipynb").is_file():
    REPO = Path("/content/intro-ds-arxiv-project")
    BRANCH = "m3-labeling"   # the Milestone 3 branch
    URL = "https://github.com/thinhelzan842/intro-ds-arxiv-project.git"
    if not (REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin",
                        f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "src/requirements.txt")], check=True)
    os.chdir(REPO / "src")
    print("Code in use:", subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"],
                                         capture_output=True, text=True).stdout.strip())

if "RUN_TEST" not in globals():
    RUN_TEST = False         # the tests of 20-26 are not run here (each notebook runs them when opened alone)
TEST_25 = True               # the offline tests of this notebook (section 4, a few seconds)
RUN_STEPS = ("data", "labels", "features", "model", "analysis", "export")   # always run in this order
RUN_LABEL_STATUS = False     # True: only the progress of the manual labels; nothing is written
RUN_STRICT_LABELS = True     # False: a dry run on the sheets done so far (statement minimums as warnings)
RUN_AUTO_FRACTION = None     # None: AUTO_FRACTION of 21; e.g. 0.10 when 22/23 take too long
RUN_FIGURES = True           # the feature histograms of 22
RUN_ABLATION = True          # ablation and leave-one-publication-out of 23
M3_ROOT = None               # another project folder (a local copy, a test folder) instead of PROJECT_ROOT
RUN_FLUSH_DRIVE = True       # Colab: flush and unmount Drive at the end, so every file is saved
globals().update(globals().get("RUN_OVERRIDES", {}))

In [ ]:
get_ipython().run_line_magic("run", "-i ./00_config.ipynb")
if M3_ROOT:   # every path of 20-26 is derived from these
    PROJECT_ROOT = Path(M3_ROOT)
    DATA_DIR, LOG_DIR, STATE_DIR = PROJECT_ROOT / "data" / "raw", PROJECT_ROOT / "logs", PROJECT_ROOT / "state"
    PAPER_LOG = LOG_DIR / "papers.csv"
    for _folder in (DATA_DIR, LOG_DIR, STATE_DIR):
        _folder.mkdir(parents=True, exist_ok=True)
    print(f"M3_ROOT: {PROJECT_ROOT}")

for _notebook in ("20_m3_data", "21_labeling", "23_model", "26_export_hf"):   # 21 also loads 22
    get_ipython().run_line_magic("run", f"-i ./{_notebook}.ipynb")

import gc
import math
import shutil
import tempfile
import unittest
from datetime import datetime, timezone

import numpy as np
from IPython.core.inputtransformer2 import TransformerManager
from IPython.display import display

STEP_ORDER = ("data", "labels", "features", "model", "analysis", "export")
SHARED_NOTEBOOKS = ("20_m3_data", "21_labeling", "22_features", "23_model", "26_export_hf", "25_run_m3")
RUN_LOG = LOG_DIR / "m3_run.json"
# One namespace: a name defined by two notebooks keeps the value of the last one loaded. The ones 21 relies on:
assert SELECTION_PATH == LABEL_DIR / "selection.csv", f"SELECTION_PATH is {SELECTION_PATH}: redefined by another notebook"
assert _FID_RE.pattern == r"(\d{4}-\d{4,5})", f"_FID_RE is {_FID_RE.pattern}: redefined by another notebook"
HAS_24 = Path("./24_analysis.ipynb").is_file()
print(f"25 loaded | project {PROJECT_ROOT} | 24_analysis: {'yes' if HAS_24 else 'not on this branch'}")

## 1. Helpers

- `run_isolated(notebook, **values)`: the code cells of a notebook, run in a namespace of their own that starts
  from `values` (used for `24`, whose names would otherwise replace those of `20`–`23`).
- `jsonable(obj)`: numpy and pandas values turned into plain Python, for `logs/m3_run.json`.
- `step_files()`: the inputs and outputs of each step; `stale_steps()` lists the outputs older than an input.
- `label_status()`: each member's sheets in `labels/manual/` against `labels/selection.csv`, then
  `validate_labels(strict=False)` of `21`.

In [ ]:
def run_isolated(notebook, **values):
    """Run the code cells of ./<notebook>.ipynb in a namespace that starts from `values`; return it."""
    path = Path(notebook if str(notebook).endswith(".ipynb") else f"./{notebook}.ipynb")
    space = {"__name__": f"isolated_{path.stem}", "get_ipython": get_ipython, "display": display, **values}
    for number, cell in enumerate(json.loads(path.read_text(encoding="utf-8"))["cells"]):
        if cell["cell_type"] == "code":
            source = TransformerManager().transform_cell("".join(cell["source"]))
            exec(compile(source, f"<{path.stem} cell {number}>", "exec"), space)
    return space


def jsonable(obj):
    """Plain Python (dict, list, str, int, float, bool, None) for json.dump."""
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, np.generic):
        obj = obj.item()
    if isinstance(obj, float) and not math.isfinite(obj):
        return None
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        return jsonable(obj.to_dict())
    if isinstance(obj, Path):
        return str(obj)
    return str(obj)


def _newest(path):
    """Modification time of a file, or of the newest file in a folder (one level); None if there is none."""
    path = Path(path)
    if path.is_file():
        return path.stat().st_mtime
    if path.is_dir():
        times = [p.stat().st_mtime for p in path.iterdir() if p.is_file()]
        return max(times) if times else None
    return None


def step_files():
    """{step: (inputs, outputs)} on Drive (analysis has no output that a later step reads)."""
    hf_predictions = [HF_DIR / m / "data" / "predictions" for m in MEMBER_RANGES]
    return {
        "labels": ([MANUAL_DIR, M3_DIR / "entries.parquet"], [LABELS_PATH, SPLIT_PATH, AUTO_LABELS_PATH]),
        "features": ([LABELS_PATH, SPLIT_PATH, M3_DIR / "entries.parquet", M3_DIR / "refs.parquet"], [PAIRS_PATH]),
        "model": ([PAIRS_PATH], [PREDICTIONS_PATH, METRICS_PATH]),
        "export": ([PREDICTIONS_PATH], hf_predictions),
    }


def stale_steps(files=None):
    """[(step, output, newer input)] for every output that is missing or older than one of its inputs."""
    stale = []
    for step, (inputs, outputs) in (step_files() if files is None else files).items():
        times = {str(p): _newest(p) for p in inputs}
        newest_input = max(((t, p) for p, t in times.items() if t is not None), default=None)
        for out in outputs:
            t = _newest(out)
            if t is None:
                stale.append((step, str(out), "missing"))
            elif newest_input and newest_input[0] > t:
                stale.append((step, str(out), newest_input[1]))
    return stale


def name_clashes(notebooks=SHARED_NOTEBOOKS, folder="."):
    """{name: {notebook: definition}} of the top-level names that two notebooks of one namespace define
    differently. `X = globals().get("X", ...)` (take the value already there) and RUN_TEST are not clashes."""
    import ast
    defs = {}
    for notebook in notebooks:
        path = Path(folder) / f"{notebook}.ipynb"
        if not path.is_file():
            continue
        for cell in json.loads(path.read_text(encoding="utf-8"))["cells"]:
            if cell["cell_type"] != "code":
                continue
            try:
                tree = ast.parse(TransformerManager().transform_cell("".join(cell["source"])))
            except SyntaxError:
                continue
            for node in tree.body:
                if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
                    names = [node.name]
                elif isinstance(node, ast.Assign):
                    names = [n.id for t in node.targets for n in ast.walk(t) if isinstance(n, ast.Name)]
                else:
                    continue
                text = ast.unparse(node)
                if "globals().get(" in text or "globals()[" in text:
                    continue
                for name in names:
                    if name.startswith("_RUN_TEST") or name == "RUN_TEST" or name.startswith("_"):
                        continue
                    defs.setdefault(name, {}).setdefault(notebook, text)
    return {name: where for name, where in defs.items() if len(set(where.values())) > 1}


def label_progress(selection=None, manual_dir=None):
    """One row per member: sheets done in labels/manual/ out of the publications of labels/selection.csv."""
    selection = read_selection() if selection is None else selection
    manual_dir = Path(manual_dir or MANUAL_DIR)
    done = set()
    names = sorted(p.name for p in manual_dir.glob("*.csv")) if manual_dir.is_dir() else []
    for name in names:
        m = _FID_RE.search(name)              # 2605-07338.csv, also "2605-07338 - Sheet1.csv"
        if m:
            done.add(m.group(1).replace("-", "."))
    rows = []
    for member, group in selection.groupby("member", sort=True):
        ids = list(group["arxiv_id"].astype(str))
        rows.append({"member": member, "done": sum(a in done for a in ids), "of": len(ids),
                     "missing": [a for a in ids if a not in done]})
    extra = sorted(done - set(selection["arxiv_id"].astype(str)))
    return pd.DataFrame(rows, columns=["member", "done", "of", "missing"]), extra


def label_status():
    """Print each member's progress and the checks of validate_labels (not strict); write nothing."""
    progress, extra = label_progress()
    print(progress.to_string(index=False))
    if extra:
        print(f"sheets in labels/manual/ that are not in labels/selection.csv: {extra}")
    entries = load_m3("entries", columns=list(BIBTEX_COLUMNS) + ["bib_source", "position"])
    check = validate_labels(None, entries, load_m3("refs"), strict=False)
    print_validation(check)
    return progress, check

## 2. Steps

One function per step, returning the numbers kept in `logs/m3_run.json`. `run_m3(steps)` runs them in the order
of `STEP_ORDER`, stops at the first failure (the later steps would read stale files), and always appends the
record, failed or not, with the stale outputs found after the run.

- `data`: `build_m3_data()` of 20, which does nothing when `data/m3/_build.json` has the same papers and parser.
- `labels`: `run_labeling()` of 21 with `strict=RUN_STRICT_LABELS` (5 publications per member, 400 entries).
- `features`: `run_features()` of 22; the pairs are dropped from memory afterwards (23 reads the file).
- `model`: `run_model()` of 23, given the entries' `bib_source` (MRR by source) and the references (the
  check of `predictions`).
- `analysis`: `run_analysis()` of 24 in its own namespace; a failed requirement is reported, not fatal.
- `export`: `export_all()` of 26, then `check_export()` of every member folder (the check run before upload).

In [ ]:
def step_data():
    record = build_m3_data()
    s = record.get("summary") or {}
    return {"signature": record.get("signature"), "built_at": record.get("built_at"), "rows": record.get("rows"),
            "papers": s.get("papers"), "eligible": s.get("eligible")}


def step_labels():
    extra = {} if RUN_AUTO_FRACTION is None else {"fraction": RUN_AUTO_FRACTION}
    s = run_labeling(strict=RUN_STRICT_LABELS, **extra)["summary"]
    keep = ("precision", "recall", "f1")
    return {"manual": s["manual"], "auto": {k: s["auto"][k] for k in ("papers", "entries", "matched", "fraction")},
            "auto_share_outside_manual": s.get("auto_share_outside_manual"), "split": s["split"],
            "auto_vs_manual": {k: s["auto_vs_manual"].get(k) for k in keep},
            "auto_vs_manual_held_out": {k: (s.get("auto_vs_manual_held_out") or {}).get(k) for k in keep},
            "params": s["params"]}


def step_features():
    pairs, _, _ = run_features(figures=RUN_FIGURES)
    entries = pairs.drop_duplicates(KEY)
    out = {"pairs": len(pairs), "entries": len(entries), "papers": int(pairs["arxiv_id"].nunique()),
           "entries_per_partition": entries["partition"].astype(str).value_counts().to_dict(),
           "positive_pairs": int(pairs["label"].sum())}
    del pairs, entries
    gc.collect()
    return out


def step_model():
    out = run_model(entries_c=load_m3("entries", columns=["arxiv_id", "citation_key", "bib_source"]),
                    refs=load_m3("refs", columns=["arxiv_id", "ref_arxiv_id"]), do_ablation=RUN_ABLATION)
    m = out["metrics"]
    return {"chosen": m["chosen"]["config"], "tau": m["chosen"]["tau"], "nil_mode_main": m["nil_mode_main"],
            "final": {part: values["all"] for part, values in m["final"].items()},
            "predictions_rows": len(out["predictions"]),
            "predictions_papers": int(out["predictions"]["arxiv_id"].nunique())}


def step_analysis():
    if not HAS_24:
        print("[25] 24_analysis.ipynb is not on this branch: analysis skipped")
        return {"status": "skipped"}
    space = run_isolated("24_analysis", RUN_TEST=False, PROJECT_ROOT=PROJECT_ROOT, MEMBER_RANGES=dict(MEMBER_RANGES))
    result = space["run_analysis"](PROJECT_ROOT)
    checks = result["checks"]
    if not result["ok"]:
        print("[25] warning: some requirements of the statement are not met (above); the run goes on")
    return {"ok": result["ok"], "status_counts": checks["status"].value_counts().to_dict(),
            "not_passed": checks[checks["status"] != "PASS"].to_dict("records")}


def step_export():
    report = export_all()
    rows = {member: check_export(HF_DIR / member, member) for member in report["members"]}
    return {"rows": rows, "export_seconds": report["seconds"],
            "papers_outside_ranges": {t: v["papers_outside_ranges"] for t, v in report["tables"].items()}}


STEPS = {"data": step_data, "labels": step_labels, "features": step_features, "model": step_model,
         "analysis": step_analysis, "export": step_export}
assert tuple(STEPS) == STEP_ORDER


def _commit():
    try:
        return subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True,
                              timeout=10).stdout.strip() or None
    except Exception:
        return None


def run_m3(steps=None, functions=None, log_path=None, files=None):
    """Run the given steps in STEP_ORDER; append the record to logs/m3_run.json; return it."""
    functions = functions or STEPS
    steps = STEP_ORDER if steps is None else tuple(steps)   # () runs nothing (it is not "every step")
    unknown = [s for s in steps if s not in functions]
    if unknown:
        raise ValueError(f"unknown steps {unknown}; the steps are {list(functions)}")
    record = {"at": to_iso(datetime.now(timezone.utc)), "commit": _commit(), "root": str(PROJECT_ROOT),
              "parser": globals().get("M3_PARSER_VERSION"), "steps_asked": list(steps), "steps": {}}
    t0 = time.time()
    try:
        for name in [s for s in functions if s in steps]:
            print(f"\n{'=' * 30} 25: {name} {'=' * 30}")
            t = time.time()
            try:
                result = functions[name]() or {}
            except Exception as err:
                record["steps"][name] = {"status": "failed", "seconds": round(time.time() - t, 1),
                                         "error": f"{type(err).__name__}: {err}"[:2000]}
                print(f"[25] {name} failed after {time.time() - t:.0f}s: the later steps are not run")
                raise
            record["steps"][name] = {"status": result.pop("status", "ok"), "seconds": round(time.time() - t, 1),
                                     **result}
    finally:
        record["seconds"] = round(time.time() - t0, 1)
        record["stale"] = [list(s) for s in stale_steps(files)]
        log_path = Path(log_path or RUN_LOG)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        old = read_json(log_path, {}) or {}
        old.setdefault("runs", []).append(jsonable(record))
        write_json(log_path, old)
    return record


def print_run(record):
    """The steps and the numbers to send to the group (and to put on the poster)."""
    rows = [{"step": n, "status": s["status"], "seconds": s["seconds"]} for n, s in record["steps"].items()]
    print(pd.DataFrame(rows).to_string(index=False))
    s = record["steps"]
    if "labels" in s and s["labels"]["status"] == "ok":
        lab = s["labels"]
        print(f"Manual: {lab['manual'].get('papers')} papers, {lab['manual'].get('entries')} entries | automatic: "
              f"{lab['auto']['papers']} papers, {lab['auto']['entries']} entries | matcher vs manual {lab['auto_vs_manual']}")
        print(f"Split: {lab['split']}")
    if "model" in s and s["model"]["status"] == "ok":
        mod = s["model"]
        print(f"Model {mod['chosen']} (tau {mod['tau']}) | {mod['predictions_rows']:,} predictions rows, "
              f"{mod['predictions_papers']} papers")
        for part, values in mod["final"].items():
            print(f"  {part}: " + ", ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}"
                                            for k, v in values.items()))
    if "export" in s and s["export"]["status"] == "ok":
        print("Exported (checked): " + "; ".join(f"{m}: " + ", ".join(f"{t} {n:,}" for t, n in r.items())
                                                for m, r in s["export"]["rows"].items()))
        print("Next: each member opens 26_export_hf with their own HF_TOKEN, UPLOAD_MEMBER = 'TVx', Run all, "
              "and the group pastes the four links on Moodle.")
    if record["stale"]:
        print("Stale outputs (older than an input: run these steps again before using their numbers):")
        for step, out, newer in record["stale"]:
            print(f"  - {step}: {out} (newer: {newer})")
    print(f"Record appended to {RUN_LOG}")

## 3. Tests

Offline, a few seconds (`TEST_25`): a failing step stops the chain and is recorded, the steps always run in
order, `stale_steps` sees a missing or older output, `run_isolated` keeps a notebook's names out of this
namespace, **no top-level name is defined differently by two notebooks of the shared namespace** (`name_clashes`:
`SELECTION_PATH` of 21 and of 23 once pointed to two files), `jsonable` turns numpy values into plain ones,
`label_progress` counts a sheet exported by Google Sheets under another name.

In [ ]:
class RunTests(unittest.TestCase):
    def setUp(self):
        self.tmp = Path(tempfile.mkdtemp(prefix="m3-run-test-"))

    def tearDown(self):
        shutil.rmtree(self.tmp, ignore_errors=True)

    def test_order_failure_and_record(self):
        calls = []

        def ok(name):
            return lambda: calls.append(name) or {"n": np.int64(3)}

        def boom():
            raise RuntimeError("bad sheet")

        functions = {"data": ok("data"), "labels": boom, "features": ok("features")}
        log = self.tmp / "run.json"
        with self.assertRaises(RuntimeError):
            run_m3(("features", "labels", "data"), functions, log, files={})
        self.assertEqual(calls, ["data"])                    # in order, and nothing after the failure
        run = read_json(log)["runs"][-1]
        self.assertEqual(run["steps"]["data"], {"status": "ok", "seconds": run["steps"]["data"]["seconds"], "n": 3})
        self.assertEqual(run["steps"]["labels"]["status"], "failed")
        self.assertIn("bad sheet", run["steps"]["labels"]["error"])
        self.assertNotIn("features", run["steps"])
        calls.clear()
        self.assertEqual(run_m3((), functions, log, files={})["steps"], {})   # an empty list runs nothing
        self.assertEqual(calls, [])
        with self.assertRaises(ValueError):
            run_m3(("nope",), functions, log, files={})

    def test_stale_steps(self):
        a, b, c = self.tmp / "a", self.tmp / "b", self.tmp / "folder"
        a.write_text("x")
        b.write_text("y")
        c.mkdir()
        (c / "sheet.csv").write_text("z")
        os.utime(a, (1000, 1000))
        os.utime(b, (2000, 2000))
        os.utime(c / "sheet.csv", (3000, 3000))
        self.assertEqual(stale_steps({"s": ([a], [b])}), [])
        self.assertEqual(stale_steps({"s": ([b], [a])}), [("s", str(a), str(b))])
        self.assertEqual(stale_steps({"s": ([c], [b])}), [("s", str(b), str(c))])          # a newer sheet
        self.assertEqual(stale_steps({"s": ([a], [self.tmp / "none"])}), [("s", str(self.tmp / "none"), "missing")])

    def test_run_isolated_keeps_names_apart(self):
        nb = {"cells": [{"cell_type": "code", "metadata": {}, "source": [
            "KEY = ['other']\n", "RANGES = globals().get('MEMBER_RANGES', {})\n", "def member_of(x):\n", "    return 'mine'\n"]}],
              "metadata": {}, "nbformat": 4, "nbformat_minor": 5}
        path = self.tmp / "toy.ipynb"
        path.write_text(json.dumps(nb), encoding="utf-8")
        before = member_of("2603.20822"), list(KEY)
        space = run_isolated(path, MEMBER_RANGES={"TVx": ("1", "2")})
        self.assertEqual(space["member_of"]("x"), "mine")
        self.assertEqual(space["RANGES"], {"TVx": ("1", "2")})
        self.assertEqual((member_of("2603.20822"), list(KEY)), before)   # this namespace is untouched
        self.assertEqual(before[0], "TV1")

    def test_no_name_defined_twice_differently(self):
        clashes = name_clashes()
        self.assertEqual(clashes, {}, "rename one of them (one namespace): "
                         + "; ".join(f"{n} in {list(w)}" for n, w in clashes.items()))
        nb = {"cells": [{"cell_type": "code", "metadata": {}, "source": ["PATH = 'a'\n"]}], "metadata": {},
              "nbformat": 4, "nbformat_minor": 5}
        for name, value in (("one", "PATH = 'a'\n"), ("two", "PATH = 'b'\n"), ("three", "PATH = globals().get('PATH', 'c')\n")):
            nb["cells"][0]["source"] = [value]
            (self.tmp / f"{name}.ipynb").write_text(json.dumps(nb), encoding="utf-8")
        self.assertEqual(set(name_clashes(("one", "two", "three"), self.tmp)), {"PATH"})
        self.assertEqual(name_clashes(("one", "three"), self.tmp), {})

    def test_jsonable(self):
        out = jsonable({"a": np.float64(0.5), "b": [np.int32(2), float("nan")], 3: pd.Series({"x": 1}), "p": Path("q")})
        self.assertEqual(out, {"a": 0.5, "b": [2, None], "3": {"x": 1}, "p": "q"})
        json.dumps(out)

    def test_label_progress(self):
        selection = pd.DataFrame({"member": ["TV1", "TV1", "TV2"], "arxiv_id": ["2603.20822", "2603.21372", "2609.33231"]})
        manual = self.tmp / "manual"
        manual.mkdir()
        (manual / "2603-20822 - Sheet1.csv").write_text("x")
        (manual / "2605-00001.csv").write_text("x")
        progress, extra = label_progress(selection, manual)
        self.assertEqual(progress.to_dict("records"), [
            {"member": "TV1", "done": 1, "of": 2, "missing": ["2603.21372"]},
            {"member": "TV2", "done": 0, "of": 1, "missing": ["2609.33231"]}])
        self.assertEqual(extra, ["2605.00001"])


if TEST_25:
    _result = unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromTestCase(RunTests))
    assert _result.wasSuccessful(), "25 tests failed"

## 4. Run

`RUN_LABEL_STATUS = True`: the progress of the manual labels only. Otherwise the steps of `RUN_STEPS`, then the
summary for the group and the poster; on Colab, Drive is flushed and unmounted at the end so that every file is
saved.

In [ ]:
if RUN_LABEL_STATUS:
    M3_LABEL_STATUS = label_status()
elif not RUN_STEPS:
    print("[25] RUN_STEPS is empty: nothing to run (the functions are loaded)")
else:
    _older = [s for s in stale_steps() if s[2] != "missing"]
    if _older:
        print("[25] outputs older than their inputs before this run (the steps run now refresh theirs):")
        for _step, _out, _newer in _older:
            print(f"  - {_step}: {Path(_out).name} older than {Path(_newer).name}")
    M3_RUN = run_m3(RUN_STEPS)
    print()
    print_run(M3_RUN)
if ON_COLAB and RUN_FLUSH_DRIVE:
    from google.colab import drive
    drive.flush_and_unmount()
    print("Drive flushed and unmounted: every file is saved")